# E2 cuối cùng — đủ bốn variants × ba seeds

A1 causal next-BCE; A2 causal next-CE; A3 causal masked-CE; A4 bidirectional masked-CE. Thay thế bản partial để viết kết luận cuối; bản partial và raw source cũ vẫn giữ nguyên. Không kết luận causal/global vượt quá matched procedure.


In [ ]:
import os, sys, json
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src/analysis/course_final.py").is_file()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))
from src.analysis.course_final import analyze, ARCHIVE_NAME, ARCHIVE_SHA256
from src.analysis.master_session1 import BUNDLE_HASH
TABLES = ROOT / "results/tables/course_final"
FULL_AUDIT = False  # True requires three private source ZIPs and the matching dataset.
REBUILD_PUBLICATION = False
if FULL_AUDIT:
    archive = Path(os.environ.get("SEQREC_FINAL_ARCHIVE", str(ROOT / "artifacts/e2_final" / ARCHIVE_NAME)))
    bundle = Path(os.environ.get("SEQREC_BUNDLE", str(ROOT / "data/processed/beauty" / BUNDLE_HASH)))
    analyze(archive, ROOT, bundle, 10000)
if REBUILD_PUBLICATION:
    from gen_fig_course_final import generate
    from write_course_final_report import write_report
    generate(ROOT)
    write_report(ROOT)
audit = json.loads((TABLES / "artifact_audit.json").read_text())
assert audit["state"] == "VERIFIED" and audit["archive_sha256"] == ARCHIVE_SHA256
assert audit["runs_verified"] == 28 and audit["prior_changed_files"] == 0
assert audit["training_closed"] and audit["remaining_requested_training_runs"] == 0
print("Raw re-audit" if FULL_AUDIT else "Public audited-table mode, not a fresh raw audit")
print("Source:", ARCHIVE_SHA256)


In [ ]:
summary = pd.read_csv(TABLES / "metric_summary.csv")
e2 = summary[summary.experiment == "E2"]
assert (e2.n_runs == 3).all() and e2.design_complete.all()
display(e2)
display(pd.read_csv(TABLES / "per_seed_metrics.csv").query("experiment == 'E2'"))


## Ba contrasts và uncertainty

10.000 paired-user resamples, sau khi lấy mean contribution trong user qua ba fitted seeds. Percentile 95%, pointwise/unadjusted; không full retraining uncertainty hoặc confirmatory p-value. BCE-vs-CE và next-vs-mask là objective packages có nhiều thay đổi về negative coverage/target density, không cô lập một scalar hoặc riêng masking.


In [ ]:
paired = pd.read_csv(TABLES / "paired_bootstrap.csv")
display(paired[paired.experiment == "E2"])
display(pd.read_csv(TABLES / "scientific_findings.csv").query("experiment == 'E2'"))


In [ ]:
display(Image(filename=str(ROOT / "results/figures/course_final/e2_complete_results.png")))
display(Image(filename=str(ROOT / "results/figures/course_final/e2_test_metrics.png")))
display(Image(filename=str(ROOT / "results/figures/course_final/e2_paired_contrasts.png")))


## Dynamics và recorded efficiency

Loss khác thang đo giữa objectives; dùng within-variant curves. Pipeline SUCCESS không phải chứng minh global convergence; ghi rõ cap/patience. Latency là encoded-history/full-catalog scoring không sorting/disk I/O, không đại diện dedicated GPU profiling.


In [ ]:
display(pd.read_csv(TABLES / "e2_training_dynamics.csv"))
display(pd.read_csv(TABLES / "runs.csv").query("experiment == 'E2'"))
display(pd.read_csv(TABLES / "e2_efficiency_summary.csv"))


In [ ]:
display(Image(filename=str(ROOT / "results/figures/course_final/e2_training_validation.png")))
display(Image(filename=str(ROOT / "results/figures/course_final/e2_recorded_efficiency.png")))
